In [ ]:
# f_clean 파일들의 통합
# lat, lng가 수치형으로 되어 있는지 검증 필요
# is_valid_location의 타입 체크 -> boolean이어야 함
# is_valid_location의 T/F 비율 및 수치 계산

In [3]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

In [6]:
# ===== 입력 파일들 (업로드된 경로) =====
files = [
    "../data/f_clean_clean_hyperblick.csv",
    "../data/f_clean_clean_koreamasa.csv",
    "../data/f_clean_gmnara.csv",
    "../data/f_clean_makangs.csv",
    "../data/f_clean_naviya.csv",
    "../data/f_clean_roomhubs_seoul.csv",
]

# ===== 기준 스키마(컬럼 순서 고정) =====
SCHEMA = [
    "name","category","address_std","gu","sigu","lat","lng","is_valid_location",
    "COL_ADM_SE","station","station_exit","station_detail"
]

def read_csv_safely(fp: str) -> pd.DataFrame:
    # UTF-8 -> CP949 -> EUC-KR 순서로 시도
    for enc in ("utf-8-sig", "utf-8", "cp949", "euc-kr"):
        try:
            return pd.read_csv(fp, encoding=enc)
        except Exception:
            pass
    # 그래도 실패하면 에러를 올려 원인 확인
    return pd.read_csv(fp)

dfs = []
for fp in files:
    df = read_csv_safely(fp)

    # 혹시 스키마가 살짝 달라도 맞춰서 결합되도록 보정
    for c in SCHEMA:
        if c not in df.columns:
            df[c] = pd.NA
    df = df[SCHEMA]

    df["source_file"] = Path(fp).name

    dfs.append(df)

merged = pd.concat(dfs, ignore_index=True)

out_path = "../data/crwaling_merged.csv"
merged.to_csv(out_path, index=False, encoding="utf-8-sig")

merged.shape, out_path

((825, 13), '../data/crwaling_merged.csv')

In [7]:
df = pd.read_csv("../data/crwaling_merged.csv")

In [8]:
df[["lat", "lng"]].dtypes

lat    float64
lng    float64
dtype: object

In [10]:
df[["lat","lng"]].isna().sum()

lat    32
lng    32
dtype: int64

In [11]:
df[["lat","lng"]].describe()

,lat,lng
count,793.000000,793.000000
mean,37.530306,127.001534
std,0.043936,0.083505
min,37.454642,126.812050
25%,37.498652,126.927141
50%,37.516135,127.028500
75%,37.557519,127.056989
max,37.680186,127.146714


In [13]:
df["is_valid_location"].dtype

dtype('bool')

In [14]:
df["is_valid_location"].value_counts(dropna=False)

is_valid_location
True     596
False    229
Name: count, dtype: int64

In [15]:
valid_counts = df["is_valid_location"].value_counts(dropna=False)
valid_ratio = df["is_valid_location"].value_counts(normalize=True, dropna=False)

valid_counts
valid_ratio

is_valid_location
True     0.722424
False    0.277576
Name: proportion, dtype: float64

((197, 13), (0, 13))